# 7주차 실습: 지도학습 (2) — 분류 알고리즘 & 정보이론
## 로지스틱 회귀 · 결정 트리 · 엔트로피 · 크로스 엔트로피 · KL Divergence

**학습 목표**
- 분류 문제의 개념과 회귀 문제와의 차이를 이해한다
- 로지스틱 회귀의 시그모이드 함수와 확률적 해석을 이해하고 구현한다
- 결정 트리의 분기 기준(지니 불순도, 엔트로피)을 이해하고 구현한다
- 혼동 행렬, 정확도, 정밀도, 재현율, F1-Score로 분류 모델을 평가한다
- **정보이론**: 엔트로피, 크로스 엔트로피, KL Divergence를 이해하고 ML과의 연결을 파악한다


## 0. 필요한 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'   # Windows
# matplotlib.rcParams['font.family'] = 'AppleGothic'   # macOS
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

print('라이브러리 임포트 완료!')
print(f'NumPy 버전: {np.__version__}')
print(f'Pandas 버전: {pd.__version__}')


---
## 1. 분류 문제(Classification)란?

**분류 문제**: 입력 데이터를 미리 정해진 **클래스(범주)** 중 하나로 예측하는 문제

| 구분 | 회귀 (Regression) | 분류 (Classification) |
|------|-------------------|----------------------|
| 출력 | 연속적인 숫자값 | 이산적인 범주(클래스) |
| 예시 | 집값 예측, 기온 예측 | 스팸 탐지, 질병 진단 |
| 대표 알고리즘 | 선형 회귀 | 로지스틱 회귀, 결정 트리 |

### 분류 유형
- **이진 분류 (Binary)**: 클래스 2개 (예: 스팸/정상)
- **다중 클래스 (Multiclass)**: 클래스 3개 이상 (예: 붓꽃 품종 3종)
- **다중 레이블 (Multi-label)**: 여러 레이블 동시 예측


In [ ]:
# 분류 문제 시각화: 두 클래스와 결정 경계
np.random.seed(42)

X0 = np.random.randn(50, 2) + [-1, -1]  # 클래스 0
X1 = np.random.randn(50, 2) + [1, 1]   # 클래스 1

plt.figure(figsize=(6, 5))
plt.scatter(X0[:, 0], X0[:, 1], color='steelblue', label='클래스 0', alpha=0.7)
plt.scatter(X1[:, 0], X1[:, 1], color='tomato',    label='클래스 1', alpha=0.7)

x_line = np.linspace(-4, 4, 100)
plt.plot(x_line, -x_line, 'k--', linewidth=2, label='결정 경계')

plt.xlabel('특성 1 (x1)')
plt.ylabel('특성 2 (x2)')
plt.title('분류 문제: 결정 경계(Decision Boundary)')
plt.legend()
plt.tight_layout()
plt.show()


---
## 2. 로지스틱 회귀 (Logistic Regression)

### 핵심 아이디어
선형 회귀 출력값은 **-inf ~ +inf** 범위 → 확률로 직접 사용 불가!  
→ **시그모이드 함수(Sigmoid Function)** 로 0~1 사이 확률값으로 변환

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

여기서 z = w1*x1 + w2*x2 + ... + b (선형 결합)

### 의사결정 규칙
- P >= 0.5 → 클래스 1
- P < 0.5  → 클래스 0


In [ ]:
# 시그모이드 함수 시각화
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
sigma = sigmoid(z)

plt.figure(figsize=(9, 4))

# 왼쪽: 시그모이드 곡선
plt.subplot(1, 2, 1)
plt.plot(z, sigma, color='darkorange', linewidth=3, label='sigma(z)')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='임계값 0.5')
plt.axvline(x=0,   color='gray', linestyle='--', alpha=0.5)
plt.fill_between(z, sigma, 0.5, where=(z >= 0), alpha=0.15, color='tomato',    label='클래스 1')
plt.fill_between(z, sigma, 0.5, where=(z <= 0), alpha=0.15, color='steelblue', label='클래스 0')
plt.xlabel('z (선형 결합 값)')
plt.ylabel('sigma(z) = P(y=1|x)')
plt.title('시그모이드 함수')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)

# 오른쪽: z 값에 따른 확률 막대
plt.subplot(1, 2, 2)
z_vals = np.array([-4, -2, -1, 0, 1, 2, 4])
p_vals = sigmoid(z_vals)
colors = ['steelblue' if p < 0.5 else 'tomato' for p in p_vals]

bars = plt.bar(range(len(z_vals)), p_vals, color=colors, alpha=0.8)
plt.axhline(y=0.5, color='black', linestyle='--', linewidth=1.5)
plt.xticks(range(len(z_vals)), [f'z={v}' for v in z_vals], fontsize=9)
plt.ylabel('확률 P(y=1|x)')
plt.title('z 값에 따른 클래스 1 확률')

for i, (bar, p) in enumerate(zip(bars, p_vals)):
    plt.text(bar.get_x() + bar.get_width()/2, p + 0.02,
             f'{p:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


### 2-1. 유방암 진단 데이터셋으로 로지스틱 회귀 실습

**데이터셋**: `load_breast_cancer()` — 유방암 양성/음성 이진 분류  
- 샘플 수: 569개  
- 특성 수: 30개 (종양의 크기, 모양, 질감 등)  
- 클래스: 악성(malignant=0) / 양성(benign=1)


In [ ]:
# 데이터 로드
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print('=== 유방암 데이터셋 기본 정보 ===')
print(f'데이터 크기: {X.shape}')
print(f'클래스: {cancer.target_names}')
print()
print('클래스 분포:')
for i, name in enumerate(cancer.target_names):
    count = np.sum(y == i)
    print(f'  {name}: {count}개 ({count/len(y)*100:.1f}%)')
print(f'\n특성 예시: {list(cancer.feature_names[:5])}')


In [ ]:
# 데이터 분리 및 표준화
# 1. 학습/테스트 분리 (항상 먼저!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify: 클래스 비율 유지
)

# 2. 표준화 (학습 데이터로 fit, 테스트 데이터는 transform만 → 데이터 누수 방지)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('=== 데이터 분리 완료 ===')
print(f'학습 데이터: {X_train_scaled.shape}')
print(f'테스트 데이터: {X_test_scaled.shape}')
print()
print(f'표준화 전 특성 범위: {X_train[:, 0].min():.2f} ~ {X_train[:, 0].max():.2f}')
print(f'표준화 후 특성 범위: {X_train_scaled[:, 0].min():.2f} ~ {X_train_scaled[:, 0].max():.2f}')


In [ ]:
# 로지스틱 회귀 모델 학습
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# 예측
y_pred_lr = lr_model.predict(X_test_scaled)          # 클래스 예측
y_prob_lr = lr_model.predict_proba(X_test_scaled)    # 확률 예측

print('=== 로지스틱 회귀 예측 결과 (첫 5개) ===')
print(f'{"인덱스":>6}  {"실제":>8}  {"예측":>8}  {"악성 확률":>10}  {"양성 확률":>10}')
print('-' * 55)
for i in range(5):
    actual    = cancer.target_names[y_test[i]]
    predicted = cancer.target_names[y_pred_lr[i]]
    print(f'{i:>6}  {actual:>8}  {predicted:>8}  {y_prob_lr[i][0]:>10.4f}  {y_prob_lr[i][1]:>10.4f}')


In [ ]:
# 분류 리포트
print('=== 로지스틱 회귀 분류 리포트 ===\n')
print(classification_report(y_test, y_pred_lr, target_names=cancer.target_names))

print(f'정확도  (Accuracy) : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'정밀도  (Precision): {precision_score(y_test, y_pred_lr):.4f}')
print(f'재현율  (Recall)   : {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1-Score           : {f1_score(y_test, y_pred_lr):.4f}')


In [ ]:
# 계수(Coefficient) 분석: 어떤 특성이 중요한가?
coefficients = lr_model.coef_[0]
feature_names = cancer.feature_names

# 절댓값 기준 상위 10개 특성
top_idx = np.argsort(np.abs(coefficients))[::-1][:10]

plt.figure(figsize=(9, 4))
colors = ['tomato' if c > 0 else 'steelblue' for c in coefficients[top_idx]]
plt.barh(range(10), coefficients[top_idx], color=colors, alpha=0.8)
plt.yticks(range(10), [feature_names[i] for i in top_idx], fontsize=9)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('계수 (Coefficient)')
plt.title('로지스틱 회귀: 특성 중요도 (상위 10개)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('빨강: 양성(benign) 방향  |  파랑: 악성(malignant) 방향')


---
## 3. 결정 트리 (Decision Tree)

### 핵심 아이디어
**스무고개처럼 질문을 반복**해서 데이터를 분류하는 알고리즘

```
꽃잎 길이 <= 2.45cm?
   |-- 예 --> Setosa (끝)
   +-- 아니오 --> 꽃잎 너비 <= 1.75cm?
                    |-- 예 --> Versicolor
                    +-- 아니오 --> Virginica
```

### 분기 기준: 불순도(Impurity) 측정

| 지표 | 수식 | 특징 |
|------|------|------|
| **지니 불순도** (Gini) | 1 - sum(pi^2) | Scikit-learn 기본값, 계산 빠름 |
| **엔트로피** (Entropy) | -sum(pi * log2(pi)) | 정보 이론 기반, 조금 더 균형적 |

두 지표 모두 **0에 가까울수록 순수한(pure) 노드**를 의미한다.


In [ ]:
# 지니 불순도와 엔트로피 직접 계산
def gini(p_list):
    """지니 불순도: p_list는 클래스별 확률 리스트"""
    return 1 - sum([pi**2 for pi in p_list])

def entropy(p_list):
    """엔트로피: p_list는 클래스별 확률 리스트"""
    return -sum([pi * np.log2(pi + 1e-10) for pi in p_list])

# 이진 분류에서 양성 비율 변화에 따른 불순도
p_range = np.linspace(0.01, 0.99, 100)
gini_vals    = [gini([p, 1-p]) for p in p_range]
entropy_vals = [entropy([p, 1-p]) / 2 for p in p_range]  # /2 스케일 맞춤

plt.figure(figsize=(7, 4))
plt.plot(p_range, gini_vals,    label='지니 불순도',        color='darkorange', linewidth=2.5)
plt.plot(p_range, entropy_vals, label='엔트로피 / 2',       color='steelblue',  linewidth=2.5, linestyle='--')
plt.axvline(x=0.5, color='gray', linestyle=':', alpha=0.7)
plt.xlabel('클래스 1의 비율 (p)')
plt.ylabel('불순도 값')
plt.title('지니 불순도 vs 엔트로피 (이진 분류)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 예시 계산
print('=== 불순도 계산 예시 ===')
examples = [
    ([1.0, 0.0], '완전 순수 노드 (전부 클래스 0)'),
    ([0.5, 0.5], '최대 혼잡 (50:50)'),
    ([0.3, 0.7], '일반적인 경우 (30:70)'),
]
print(f'{"분포":>20}  {"지니":>8}  {"엔트로피":>10}  설명')
print('-' * 60)
for probs, desc in examples:
    g = gini(probs)
    e = entropy(probs)
    print(f'{str(probs):>20}  {g:>8.4f}  {e:>10.4f}  {desc}')


### 3-1. 붓꽃(Iris) 데이터셋으로 결정 트리 실습

**데이터셋**: `load_iris()` — 붓꽃 3종 분류  
- 샘플 수: 150개  
- 특성 수: 4개 (꽃받침 길이/너비, 꽃잎 길이/너비)  
- 클래스: Setosa / Versicolor / Virginica


In [ ]:
# 데이터 로드
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print('=== 붓꽃 데이터셋 기본 정보 ===')
print(f'데이터 크기: {X_iris.shape}')
print(f'클래스: {iris.target_names}')
print(f'특성: {list(iris.feature_names)}')

df_iris = pd.DataFrame(X_iris, columns=iris.feature_names)
df_iris['species'] = [iris.target_names[i] for i in y_iris]
print('\n첫 5행:')
print(df_iris.head())


In [ ]:
# 학습/테스트 분리 (결정 트리는 스케일 정규화 불필요!)
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

print(f'학습 데이터: {X_train_i.shape}')
print(f'테스트 데이터: {X_test_i.shape}')
print('\n결정 트리는 특성 스케일에 무관하므로 표준화 불필요!')


In [ ]:
# 결정 트리 모델 학습
dt_model = DecisionTreeClassifier(
    max_depth=3,        # 최대 깊이 제한 (과적합 방지)
    criterion='gini',   # 분기 기준: 'gini' 또는 'entropy'
    random_state=42
)
dt_model.fit(X_train_i, y_train_i)

y_pred_dt = dt_model.predict(X_test_i)

print('=== 결정 트리 분류 리포트 ===\n')
print(classification_report(y_test_i, y_pred_dt, target_names=iris.target_names))
print(f'정확도: {accuracy_score(y_test_i, y_pred_dt):.4f}')


In [ ]:
# 결정 트리 구조 텍스트 출력
print('=== 결정 트리 구조 (텍스트) ===\n')
tree_rules = export_text(dt_model, feature_names=list(iris.feature_names))
print(tree_rules)


In [ ]:
# 결정 트리 시각화
plt.figure(figsize=(14, 6))
plot_tree(
    dt_model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,    # 클래스별 색상 채우기
    rounded=True,   # 모서리 둥글게
    fontsize=10
)
plt.title('결정 트리 시각화 (max_depth=3, Iris 데이터셋)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# 특성 중요도 (Feature Importance)
importances  = dt_model.feature_importances_
sorted_idx   = np.argsort(importances)[::-1]

print('=== 특성 중요도 ===')
for i in sorted_idx:
    bar = '#' * int(importances[i] * 40)
    print(f'  {iris.feature_names[i]:30s}: {importances[i]:.4f}  {bar}')

plt.figure(figsize=(7, 3.5))
colors = ['#E07B39' if i == sorted_idx[0] else '#417D7A' for i in range(4)]
plt.bar(range(4), importances[sorted_idx], color=colors, alpha=0.85)
plt.xticks(range(4), [iris.feature_names[i] for i in sorted_idx],
           rotation=15, ha='right', fontsize=9)
plt.ylabel('중요도 (Gini Importance)')
plt.title('결정 트리: 특성 중요도')
plt.tight_layout()
plt.show()


---
## 4. 과적합(Overfitting)과 가지치기(Pruning)

결정 트리는 깊이 제한 없이 학습하면 **훈련 데이터에 과적합**될 수 있다.

| 파라미터 | 설명 | 권장값 |
|----------|------|--------|
| `max_depth` | 트리 최대 깊이 | 3~5 |
| `min_samples_split` | 노드 분기 최소 샘플 수 | 2~10 |
| `min_samples_leaf` | 리프 노드 최소 샘플 수 | 1~5 |


In [ ]:
# max_depth 변화에 따른 훈련/테스트 정확도 비교
depths       = range(1, 12)
train_scores = []
test_scores  = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_i, y_train_i)
    train_scores.append(accuracy_score(y_train_i, dt.predict(X_train_i)))
    test_scores.append(accuracy_score(y_test_i,   dt.predict(X_test_i)))

best_depth = list(depths)[np.argmax(test_scores)]
best_score = max(test_scores)

plt.figure(figsize=(8, 5))
plt.plot(list(depths), train_scores, 'o-', color='tomato',    linewidth=2.5, markersize=7, label='훈련 정확도')
plt.plot(list(depths), test_scores,  's-', color='steelblue', linewidth=2.5, markersize=7, label='테스트 정확도')
plt.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7)
plt.annotate(f'최적 깊이={best_depth}\n정확도={best_score:.3f}',
             xy=(best_depth, best_score),
             xytext=(best_depth + 1.0, best_score - 0.05),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=9, color='green')
plt.xlabel('트리 깊이 (max_depth)')
plt.ylabel('정확도')
plt.title('트리 깊이와 과적합: 훈련 vs 테스트 정확도')
plt.legend()
plt.xticks(list(depths))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'최적 max_depth: {best_depth}  (테스트 정확도: {best_score:.4f})')
print('깊이가 깊을수록 훈련 정확도 상승, 하지만 테스트 정확도는 일정 이후 감소 -> 과적합!')


---
## 5. 분류 모델 평가 지표

### 혼동 행렬 (Confusion Matrix)

|  | 예측: Positive | 예측: Negative |
|--|---------------|---------------|
| **실제: Positive** | TP (진짜 양성) | FN (거짓 음성) |
| **실제: Negative** | FP (거짓 양성) | TN (진짜 음성) |

### 평가 지표

| 지표 | 공식 | 의미 |
|------|------|------|
| **정확도** (Accuracy)  | (TP+TN) / 전체 | 전체 중 올바른 예측 비율 |
| **정밀도** (Precision) | TP / (TP+FP)   | 양성 예측 중 실제 양성 비율 |
| **재현율** (Recall)    | TP / (TP+FN)   | 실제 양성 중 양성으로 예측한 비율 |
| **F1-Score**           | 2*P*R / (P+R)  | 정밀도와 재현율의 조화 평균 |

> **상황에 따른 지표 선택**  
> - 스팸 탐지: **정밀도** 중요 (정상 메일을 스팸으로 오분류 방지)  
> - 암 진단: **재현율** 중요 (환자를 정상으로 오분류 방지)


In [ ]:
# 혼동 행렬 시각화 비교 (유방암 데이터)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 로지스틱 회귀
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm_lr, display_labels=cancer.target_names).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('로지스틱 회귀 -- 혼동 행렬', fontsize=12)

# 결정 트리 (유방암 데이터로 학습)
dt_cancer = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_cancer.fit(X_train_scaled, y_train)
y_pred_dt_cancer = dt_cancer.predict(X_test_scaled)

cm_dt = confusion_matrix(y_test, y_pred_dt_cancer)
ConfusionMatrixDisplay(cm_dt, display_labels=cancer.target_names).plot(
    ax=axes[1], colorbar=False, cmap='Oranges'
)
axes[1].set_title('결정 트리 -- 혼동 행렬', fontsize=12)

plt.tight_layout()
plt.show()

# 두 모델 성능 비교 표
print('\n=== 유방암 데이터셋: 두 모델 성능 비교 ===')
print(f'{"지표":>12}  {"로지스틱 회귀":>14}  {"결정 트리":>12}')
print('-' * 44)

for name, func in [('정확도', accuracy_score), ('정밀도', precision_score),
                   ('재현율', recall_score),    ('F1-Score', f1_score)]:
    kwargs = {} if name == '정확도' else {}
    lr_val = func(y_test, y_pred_lr) if name == '정확도' else func(y_test, y_pred_lr)
    dt_val = func(y_test, y_pred_dt_cancer) if name == '정확도' else func(y_test, y_pred_dt_cancer)
    better = '<-- LR 우세' if lr_val >= dt_val else '<-- DT 우세'
    print(f'{name:>12}  {lr_val:>14.4f}  {dt_val:>12.4f}  {better}')


---
## 6. 결정 경계 비교: 로지스틱 회귀 vs 결정 트리

두 알고리즘이 학습하는 **결정 경계(Decision Boundary)** 의 형태가 다르다:
- **로지스틱 회귀**: 선형 경계 (직선/평면)
- **결정 트리**: 비선형 경계 (계단형)


In [ ]:
from sklearn.datasets import make_moons

# 비선형 데이터 생성 (초승달 모양)
X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=42)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_moon, y_moon, test_size=0.2, random_state=42
)

scaler_moon = StandardScaler()
X_tr_ms = scaler_moon.fit_transform(X_tr_m)
X_te_ms = scaler_moon.transform(X_te_m)

lr_moon = LogisticRegression(random_state=42)
dt_moon = DecisionTreeClassifier(max_depth=5, random_state=42)

lr_moon.fit(X_tr_ms, y_tr_m)
dt_moon.fit(X_tr_m,  y_tr_m)

def plot_boundary(model, X, y, ax, title, sc=None):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_in = sc.transform(grid) if sc else grid
    Z = model.predict(grid_in).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', s=25, alpha=0.8, label='클래스 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='tomato',    s=25, alpha=0.8, label='클래스 1')
    te_in = sc.transform(X_te_m) if sc else X_te_m
    acc = accuracy_score(y_te_m, model.predict(te_in))
    ax.set_title(f'{title}\n테스트 정확도: {acc:.3f}')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(lr_moon, X_moon, y_moon, axes[0], '로지스틱 회귀 (선형 경계)', sc=scaler_moon)
plot_boundary(dt_moon, X_moon, y_moon, axes[1], '결정 트리 (계단형 경계)')
plt.suptitle('결정 경계 비교: 초승달 데이터 (비선형)', fontsize=13)
plt.tight_layout()
plt.show()

print('로지스틱 회귀: 직선 경계 -> 비선형 데이터에서 성능 한계')
print('결정 트리: 계단형 경계 -> 비선형 패턴 학습 가능, 단 과적합 위험')


---
## 7. 연습 문제

아래 코드를 완성하고 결과를 분석해보세요.


### 연습 문제 1
와인 품질 데이터셋(`load_wine()`)으로 로지스틱 회귀와 결정 트리를 학습하고 성능을 비교하세요.


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print(f'데이터 크기: {X_wine.shape}')
print(f'클래스: {wine.target_names}')

# TODO: 학습/테스트 분리 (test_size=0.2, random_state=42)
# X_train_w, X_test_w, y_train_w, y_test_w = ...

# TODO: 표준화
# scaler_w = StandardScaler()
# X_train_ws = ...
# X_test_ws  = ...

# TODO: 로지스틱 회귀 학습 및 정확도 출력
# lr_wine = LogisticRegression(max_iter=1000, random_state=42)
# ...
# print(f'로지스틱 회귀 정확도: ...')

# TODO: 결정 트리 학습 및 정확도 출력 (max_depth=5)
# dt_wine = DecisionTreeClassifier(max_depth=5, random_state=42)
# ...
# print(f'결정 트리 정확도: ...')


### 연습 문제 2
Iris 데이터에서 `criterion='entropy'`와 `criterion='gini'`의 테스트 정확도를 비교하세요.


In [ ]:
# TODO: 두 모델 학습 및 정확도 비교
# dt_gini    = DecisionTreeClassifier(criterion='gini',    max_depth=3, random_state=42)
# dt_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)

# (X_train_i, y_train_i, X_test_i, y_test_i 는 위에서 정의된 변수 사용)
# ...
# print(f'Gini    정확도: ...')
# print(f'Entropy 정확도: ...')


### 연습 문제 3
유방암 데이터에서 결정 트리의 `max_depth`를 1~20으로 바꾸면서
테스트 정확도가 최대가 되는 `max_depth` 값을 찾아보세요.


In [ ]:
# TODO: 최적 max_depth 탐색
# best_depth = None
# best_acc = 0
# for d in range(1, 21):
#     dt_temp = DecisionTreeClassifier(max_depth=d, random_state=42)
#     dt_temp.fit(X_train_scaled, y_train)
#     acc = accuracy_score(y_test, dt_temp.predict(X_test_scaled))
#     if acc > best_acc:
#         best_acc = acc
#         best_depth = d
# print(f'최적 max_depth: {best_depth}  정확도: {best_acc:.4f}')


---
## 8. 정보이론 기초 (Information Theory)

분류 알고리즘(결정 트리의 분기 기준, 로지스틱 회귀의 손실 함수)을 수학적으로 이해하려면
**정보이론(Information Theory)** 의 세 가지 핵심 개념이 필요하다.

| 개념 | 수식 | 핵심 의미 |
|------|------|-----------|
| **엔트로피 H(X)** | -sum p·log p | 분포의 불확실성(무질서도) |
| **크로스 엔트로피 H(P,Q)** | -sum P·log Q | 두 분포의 차이 (손실 함수) |
| **KL Divergence D_KL(P\|\|Q)** | sum P·log(P/Q) | P 대비 Q의 정보 손실량 |

세 지표의 관계:  
$$D_{KL}(P \| Q) = H(P, Q) - H(P)$$


### 8-1. 엔트로피 (Shannon Entropy)

**정보량(surprise)**: 사건 x가 발생했을 때의 정보량 = -log p(x)
- 확률이 낮을수록 (희귀할수록) 정보량이 크다
- 확률이 높을수록 (흔할수록) 정보량이 작다

**엔트로피**: 모든 사건의 정보량의 **기댓값** (평균 불확실성)

$$H(X) = -\sum_{x} p(x) \log_2 p(x)$$

- 값이 **클수록** 불확실성이 높다 (예측하기 어렵다)
- 값이 **작을수록** (0에 가까울수록) 확실하다 (한 클래스가 지배적)
- **결정 트리**: 엔트로피가 가장 많이 감소하는 방향으로 분기!


In [ ]:
# ── 자기 정보량(Self-Information) 시각화 ────────────────────────────────────
p_vals = np.linspace(0.01, 0.99, 200)
info_vals = -np.log2(p_vals)  # 자기 정보량

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. 자기 정보량 I(x) = -log2 p(x)
axes[0].plot(p_vals, info_vals, color='steelblue', linewidth=2.5)
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.6)
axes[0].scatter([0.5], [-np.log2(0.5)], color='tomato', s=80, zorder=5,
               label=f'p=0.5, I={-np.log2(0.5):.1f} bit')
axes[0].scatter([0.1], [-np.log2(0.1)], color='darkorange', s=80, zorder=5,
               label=f'p=0.1, I={-np.log2(0.1):.2f} bit')
axes[0].set_xlabel('확률 p')
axes[0].set_ylabel('자기 정보량 (bit)')
axes[0].set_title('자기 정보량  I(x) = -log2 p(x)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 2. 이진 분포의 엔트로피 H(p)
h_vals = -(p_vals * np.log2(p_vals + 1e-12) + (1-p_vals) * np.log2(1-p_vals + 1e-12))
axes[1].plot(p_vals, h_vals, color='darkorange', linewidth=2.5)
axes[1].axvline(x=0.5, color='gray', linestyle='--', alpha=0.6)
axes[1].scatter([0.5], [1.0], color='tomato', s=100, zorder=5, label='최대 H=1 (p=0.5)')
axes[1].scatter([0.1], [-(0.1*np.log2(0.1)+0.9*np.log2(0.9))],
               color='steelblue', s=80, zorder=5,
               label=f'p=0.1, H={-(0.1*np.log2(0.1)+0.9*np.log2(0.9)):.3f}')
axes[1].set_xlabel('클래스 1의 비율 p')
axes[1].set_ylabel('엔트로피 H (bit)')
axes[1].set_title('이진 분포 엔트로피  H(p)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# 3. 다양한 분포의 엔트로피 비교
distributions = {
    '균등 분포\n(최대 불확실)': [0.25, 0.25, 0.25, 0.25],
    '약간 편향\n(중간)': [0.5, 0.3, 0.1, 0.1],
    '강한 편향\n(거의 확실)': [0.9, 0.05, 0.03, 0.02],
    '완전 편향\n(확실)': [1.0, 0.0, 0.0, 0.0],
}

names = list(distributions.keys())
h_list = []
for probs in distributions.values():
    h = -sum(p * np.log2(p + 1e-12) for p in probs)
    h_list.append(h)

bar_colors = ['#E07B39', '#417D7A', '#84B59F', '#9DC5BB']
bars = axes[2].bar(range(len(names)), h_list, color=bar_colors, alpha=0.85)
axes[2].set_xticks(range(len(names)))
axes[2].set_xticklabels(names, fontsize=8.5)
axes[2].set_ylabel('엔트로피 H (bit)')
axes[2].set_title('분포 형태에 따른 엔트로피')
axes[2].set_ylim(0, 2.5)
for bar, h in zip(bars, h_list):
    axes[2].text(bar.get_x() + bar.get_width()/2, h + 0.05,
                f'{h:.3f}', ha='center', va='bottom', fontsize=10)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('정보량과 엔트로피', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# 수치 확인
print('=== 다양한 분포의 엔트로피 ===')
for name, probs in distributions.items():
    h = -sum(p * np.log2(p + 1e-12) for p in probs)
    name_clean = name.replace('\n', ' ')
    print(f'  {name_clean:20s}: H = {h:.4f} bit  (분포: {probs})')


In [ ]:
# ── 결정 트리의 엔트로피 기반 분기: 정보 이득(Information Gain) ─────────────
def entropy(probs):
    """엔트로피 계산 (0 확률 처리 포함)"""
    return -sum(p * np.log2(p + 1e-12) for p in probs if p > 0)

def information_gain(parent_probs, children):
    """
    정보 이득 = 부모 엔트로피 - 가중 평균 자식 엔트로피
    parent_probs: 부모 노드의 클래스 분포
    children: [(n_samples, [probs]), ...] 리스트
    """
    H_parent = entropy(parent_probs)
    total = sum(n for n, _ in children)
    H_children = sum((n / total) * entropy(probs) for n, probs in children)
    return H_parent - H_children

# 예시: 날씨 데이터로 분기 기준 선택
print('=== 정보 이득(Information Gain) 계산 예시 ===')
print('데이터: 14개 샘플, 9개 양성(Play=Yes), 5개 음성(Play=No)\n')

parent = [9/14, 5/14]  # 부모 노드
H_parent = entropy(parent)
print(f'부모 노드 엔트로피: H = {H_parent:.4f} bit\n')

# 특성 A로 분기: [6Y,2N] | [3Y,3N] (정보이득 높음)
gain_A = information_gain(parent, [(8, [6/8, 2/8]), (6, [3/6, 3/6])])

# 특성 B로 분기: [5Y,2N] | [4Y,3N] (정보이득 낮음)
gain_B = information_gain(parent, [(7, [5/7, 2/7]), (7, [4/7, 3/7])])

print(f'특성 A로 분기: 정보 이득 = {gain_A:.4f} bit  ← 더 좋은 분기!')
print(f'특성 B로 분기: 정보 이득 = {gain_B:.4f} bit')
print(f'\n→ 결정 트리는 정보 이득이 가장 큰 특성으로 분기를 선택!')


### 8-2. 크로스 엔트로피 (Cross Entropy)

실제 분포 **P** 를 예측 분포 **Q** 로 표현할 때 필요한 **평균 코드 길이**

$$H(P, Q) = -\sum_x P(x) \log Q(x)$$

**핵심 성질**
- P = Q 일 때: H(P,Q) = H(P) — 최솟값 (완벽한 예측)
- P ≠ Q 일 때: H(P,Q) > H(P) — 항상 엔트로피보다 크거나 같음
- **비대칭**: H(P,Q) ≠ H(Q,P)

**머신러닝 손실 함수 연결**

이진 분류에서 P = 실제 레이블(0 or 1), Q = 모델 예측 확률(sigmoid 출력)이면:

$$H(P, Q) = -[y \log q + (1-y) \log (1-q)]$$

→ 이것이 바로 **Binary Cross Entropy Loss** (로지스틱 회귀 손실 함수)!


In [ ]:
# ── 크로스 엔트로피 직접 계산 ────────────────────────────────────────────────
def cross_entropy(P, Q):
    """H(P, Q) = -sum P * log Q"""
    return -sum(p * np.log(q + 1e-12) for p, q in zip(P, Q))

def binary_cross_entropy(y_true, y_pred):
    """이진 크로스 엔트로피 손실"""
    return -np.mean(y_true * np.log(y_pred + 1e-12) +
                   (1 - y_true) * np.log(1 - y_pred + 1e-12))

# 예시 1: 다중 클래스 크로스 엔트로피
print('=== 다중 클래스 크로스 엔트로피 ===')
P_true = [1.0, 0.0, 0.0]       # 실제: 클래스 0
Q_good = [0.9, 0.05, 0.05]     # 좋은 예측
Q_bad  = [0.1, 0.6,  0.3 ]     # 나쁜 예측
Q_rand = [1/3, 1/3,  1/3 ]     # 랜덤 예측

print(f'실제 분포 P        : {P_true}')
print(f'좋은 예측 Q_good   : {Q_good}  -> H(P,Q) = {cross_entropy(P_true, Q_good):.4f}')
print(f'나쁜 예측 Q_bad    : {Q_bad}   -> H(P,Q) = {cross_entropy(P_true, Q_bad):.4f}')
print(f'랜덤 예측 Q_rand   : {Q_rand}  -> H(P,Q) = {cross_entropy(P_true, Q_rand):.4f}')
print(f'완벽 예측 Q=P      : {P_true} -> H(P,Q) = {cross_entropy(P_true, P_true):.4f}')

# 예시 2: 이진 크로스 엔트로피 손실 곡선
print('\n=== 이진 크로스 엔트로피 손실 곡선 ===')
q_range = np.linspace(0.01, 0.99, 200)

# y=1 (양성 샘플) 일 때의 손실
loss_y1 = -np.log(q_range)         # y=1: -log(q)
# y=0 (음성 샘플) 일 때의 손실
loss_y0 = -np.log(1 - q_range)     # y=0: -log(1-q)

plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.plot(q_range, loss_y1, color='tomato',    linewidth=2.5, label='y=1: loss = -log(q)')
plt.plot(q_range, loss_y0, color='steelblue', linewidth=2.5, label='y=0: loss = -log(1-q)')
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.6)
plt.xlabel('예측 확률 q')
plt.ylabel('손실 (Loss)')
plt.title('이진 크로스 엔트로피 손실')
plt.ylim(0, 5)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# 예측이 틀릴수록 손실이 폭발적으로 증가
# y=1인데 q=0.01 예측 시 손실
bad_preds  = np.array([0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99])
loss_wrong = -np.log(bad_preds)  # y=1 기준

plt.bar(range(len(bad_preds)), loss_wrong,
        color=['tomato' if p < 0.5 else 'steelblue' for p in bad_preds], alpha=0.8)
plt.xticks(range(len(bad_preds)), [f'q={p}' for p in bad_preds], rotation=20, fontsize=9)
plt.ylabel('손실 (Loss)')
plt.title('y=1 샘플의 예측 확률별 손실 크기')
plt.grid(True, alpha=0.3, axis='y')
for i, loss in enumerate(loss_wrong):
    plt.text(i, loss + 0.05, f'{loss:.2f}', ha='center', fontsize=8.5)

plt.tight_layout()
plt.show()

print('-> 정답 클래스의 확률이 낮을수록 손실이 폭발적으로 증가!')
print('-> 따라서 모델은 손실을 줄이기 위해 정답 클래스 확률을 높이는 방향으로 학습됨')


In [ ]:
# ── Scikit-learn의 로지스틱 회귀는 실제로 크로스 엔트로피를 최소화한다 ───────
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

y_prob = lr.predict_proba(X_test_s)[:, 1]  # 양성 확률

# 손실 직접 계산
bce_manual = binary_cross_entropy(y_test, y_prob)

# sklearn의 log_loss와 비교
from sklearn.metrics import log_loss
bce_sklearn = log_loss(y_test, y_prob)

print('=== 크로스 엔트로피 손실 검증 ===')
print(f'직접 계산한 BCE Loss  : {bce_manual:.6f}')
print(f'sklearn log_loss      : {bce_sklearn:.6f}')
print(f'\n정확도: {lr.score(X_test_s, y_test):.4f}')
print('\n-> 로지스틱 회귀는 내부적으로 BCE Loss를 최소화하도록 파라미터(w, b)를 학습!')


### 8-3. KL Divergence (쿨백-라이블러 발산)

두 확률 분포 P와 Q가 **얼마나 다른지** 측정하는 지표

$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

**엔트로피와의 관계**:
$$D_{KL}(P \| Q) = H(P, Q) - H(P)$$

> 크로스 엔트로피에서 엔트로피(상수)를 빼면 KL Divergence

**핵심 성질**
- **항상 비음수**: D_KL(P||Q) ≥ 0  (P=Q 일 때만 0)
- **비대칭**: D_KL(P||Q) ≠ D_KL(Q||P)  → 진짜 거리(distance metric)가 아님
- **머신러닝**: 손실 최소화 = KL Divergence 최소화 (H(P)는 상수이므로)


In [ ]:
# ── KL Divergence 직접 계산 및 비대칭성 확인 ─────────────────────────────────
def kl_divergence(P, Q):
    """D_KL(P||Q) = sum P * log(P/Q)"""
    return sum(p * np.log((p + 1e-12) / (q + 1e-12)) for p, q in zip(P, Q) if p > 0)

print('=== KL Divergence 계산 예시 ===')

# 예시 분포들
P = [0.4, 0.3, 0.2, 0.1]
Q1 = [0.35, 0.32, 0.22, 0.11]  # P와 가까운 Q
Q2 = [0.1,  0.4,  0.4,  0.1 ]  # P와 먼 Q
Q3 = [0.25, 0.25, 0.25, 0.25]  # 균등 분포

print(f'P              : {P}')
print()
print(f'Q1 (P와 유사)   : {Q1}')
print(f'  D_KL(P||Q1) = {kl_divergence(P, Q1):.6f}  (작음: 비슷한 분포)')
print()
print(f'Q2 (P와 다름)   : {Q2}')
print(f'  D_KL(P||Q2) = {kl_divergence(P, Q2):.6f}  (큼: 다른 분포)')
print()
print(f'Q3 (균등 분포)  : {Q3}')
print(f'  D_KL(P||Q3) = {kl_divergence(P, Q3):.6f}')

# 비대칭성 확인
print('\n=== 비대칭성 확인 ===')
print(f'D_KL(P||Q2) = {kl_divergence(P, Q2):.6f}')
print(f'D_KL(Q2||P) = {kl_divergence(Q2, P):.6f}')
print('-> KL Divergence는 비대칭! P와 Q를 바꾸면 값이 달라진다')


In [ ]:
# ── 세 지표의 관계 시각화: H(P), H(P,Q), D_KL(P||Q) ─────────────────────────
import matplotlib.patches as mpatches

# P를 고정하고 Q가 변할 때 세 지표 변화
P_fixed = [0.7, 0.3]  # 고정 분포
H_P = entropy(P_fixed)

# Q = [q, 1-q] 로 q를 0.01~0.99 변화
q_range = np.linspace(0.01, 0.99, 200)
H_PQ_vals = [cross_entropy(P_fixed, [q, 1-q]) for q in q_range]
H_Q_vals  = [entropy([q, 1-q]) for q in q_range]
KL_vals   = [kl_divergence(P_fixed, [q, 1-q]) for q in q_range]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 세 지표 한 그래프에
axes[0].plot(q_range, H_PQ_vals, color='darkorange',  linewidth=2.5, label='H(P,Q) 크로스 엔트로피')
axes[0].plot(q_range, KL_vals,   color='steelblue',   linewidth=2.5, label='D_KL(P||Q) KL Divergence', linestyle='--')
axes[0].axhline(y=H_P, color='green', linewidth=2, linestyle=':', label=f'H(P) = {H_P:.3f} (고정)')

# P와 Q가 같아지는 지점 (q=0.7)
q_match = 0.7
axes[0].axvline(x=q_match, color='red', linestyle='--', alpha=0.5)
axes[0].scatter([q_match], [H_P], color='red', s=100, zorder=5,
               label=f'Q=P 일 때 (q={q_match}): 최솟값')
axes[0].set_xlabel('Q의 파라미터 q')
axes[0].set_ylabel('값')
axes[0].set_title('P=[0.7, 0.3] 고정, Q=[q, 1-q] 변화')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.1, 3.0)

# 오른쪽: 관계 수식 시각화
axes[1].plot(q_range, H_PQ_vals, color='darkorange', linewidth=3, label='H(P,Q)', alpha=0.9)
axes[1].fill_between(q_range, H_P, H_PQ_vals,
                     where=[h >= H_P for h in H_PQ_vals],
                     alpha=0.25, color='steelblue', label='D_KL = H(P,Q) - H(P)')
axes[1].axhline(y=H_P, color='green', linewidth=2, linestyle=':', label=f'H(P)={H_P:.3f}')
axes[1].set_xlabel('Q의 파라미터 q')
axes[1].set_ylabel('값')
axes[1].set_title('D_KL(P||Q) = H(P,Q) - H(P)\n(파란 영역이 KL Divergence)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.1, 3.0)

plt.suptitle('엔트로피 / 크로스 엔트로피 / KL Divergence 관계', fontsize=13)
plt.tight_layout()
plt.show()

print(f'H(P)            = {H_P:.4f}  (고정, P의 불확실성)')
print(f'H(P,Q) at Q=P   = {cross_entropy(P_fixed, P_fixed):.4f}  (최솟값, Q=P 일 때)')
print(f'D_KL at Q=P     = {kl_divergence(P_fixed, P_fixed):.4f}  (0, 완벽히 같은 분포)')


In [ ]:
# ── 머신러닝에서 KL Divergence: 학습 과정 시뮬레이션 ────────────────────────
# 로지스틱 회귀 학습이 KL Divergence를 줄이는 과정

np.random.seed(42)

# 간단한 이진 분류 데이터
from sklearn.datasets import make_classification
X_sim, y_sim = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                   random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_sim, y_sim, test_size=0.3, random_state=42)

# C (정규화 강도 역수) 변화에 따른 손실 추적
C_vals     = np.logspace(-3, 3, 30)
bce_train  = []
bce_test   = []
kl_train   = []

for C in C_vals:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)

    p_train = model.predict_proba(X_tr)[:, 1]
    p_test  = model.predict_proba(X_te)[:, 1]

    bce_tr = log_loss(y_tr, p_train)
    bce_te = log_loss(y_te, p_test)

    # KL divergence (경험 분포 기준)
    # 실제 레이블을 P, 예측 확률을 Q로 볼 때
    H_empirical = entropy([np.mean(y_tr), 1 - np.mean(y_tr)])
    kl_tr = bce_tr - H_empirical  # H(P,Q) - H(P) = D_KL

    bce_train.append(bce_tr)
    bce_test.append(bce_te)
    kl_train.append(max(kl_tr, 0))  # 수치 오차 방지

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 왼쪽: BCE Loss
axes[0].semilogx(C_vals, bce_train, 'o-', color='tomato',    linewidth=2,
                 markersize=5, label='훈련 BCE Loss')
axes[0].semilogx(C_vals, bce_test,  's-', color='steelblue', linewidth=2,
                 markersize=5, label='테스트 BCE Loss')
axes[0].set_xlabel('C (정규화 강도 역수, log 스케일)')
axes[0].set_ylabel('Binary Cross Entropy Loss')
axes[0].set_title('C 값에 따른 크로스 엔트로피 손실')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 오른쪽: KL Divergence
axes[1].semilogx(C_vals, kl_train, '^-', color='darkorange', linewidth=2,
                 markersize=5, label='훈련 KL Divergence')
axes[1].set_xlabel('C (정규화 강도 역수, log 스케일)')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title('C 값에 따른 KL Divergence\n(BCE - H(P), 낮을수록 예측 분포가 실제에 가까움)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_C_idx = np.argmin(bce_test)
print(f'최적 C (테스트 BCE 기준): C = {C_vals[best_C_idx]:.4f}')
print(f'  훈련 BCE Loss: {bce_train[best_C_idx]:.4f}')
print(f'  테스트 BCE Loss: {bce_test[best_C_idx]:.4f}')
print(f'  KL Divergence: {kl_train[best_C_idx]:.4f}')


### 8-4. 정보이론 연습 문제


In [ ]:
# ─── 연습 문제 IT-1 ──────────────────────────────────────────────────────────
# 아래 세 분포 중 엔트로피가 가장 높은 것은 무엇인가요?
# 직접 계산하고 이유를 설명해보세요.

dist_A = [0.5, 0.5]                    # 이진, 균등
dist_B = [0.9, 0.1]                    # 이진, 편향
dist_C = [0.25, 0.25, 0.25, 0.25]     # 4클래스, 균등
dist_D = [0.7, 0.1, 0.1, 0.1]         # 4클래스, 편향

# TODO: entropy() 함수를 사용해 각 분포의 엔트로피를 계산하세요
# print(f'H(A) = ...')
# print(f'H(B) = ...')
# print(f'H(C) = ...')
# print(f'H(D) = ...')


In [ ]:
# ─── 연습 문제 IT-2 ──────────────────────────────────────────────────────────
# 다음 두 예측 결과 중 크로스 엔트로피 손실이 더 낮은 것은?
# 왜 그런지 설명해보세요.

# 실제 레이블 (3개 샘플, 클래스 0/1/2)
y_true_oh = [[1,0,0], [0,1,0], [0,0,1]]  # 원-핫 인코딩

# 두 모델의 예측 확률
pred_model1 = [[0.8, 0.1, 0.1],  # 샘플 1: 클래스 0 높게 예측 (정답 O)
               [0.2, 0.6, 0.2],  # 샘플 2: 클래스 1 높게 예측 (정답 O)
               [0.1, 0.2, 0.7]]  # 샘플 3: 클래스 2 높게 예측 (정답 O)

pred_model2 = [[0.5, 0.3, 0.2],  # 샘플 1: 확신이 낮은 예측
               [0.3, 0.4, 0.3],  # 샘플 2: 확신이 낮은 예측
               [0.2, 0.3, 0.5]]  # 샘플 3: 확신이 낮은 예측

# TODO: 각 모델의 크로스 엔트로피 손실을 계산하세요
# loss1 = np.mean([cross_entropy(p, q) for p, q in zip(y_true_oh, pred_model1)])
# loss2 = np.mean([cross_entropy(p, q) for p, q in zip(y_true_oh, pred_model2)])
# print(f'Model 1 CE Loss: {loss1:.4f}')
# print(f'Model 2 CE Loss: {loss2:.4f}')


In [ ]:
# ─── 연습 문제 IT-3 ──────────────────────────────────────────────────────────
# D_KL(P||Q)와 D_KL(Q||P)의 차이를 계산하고
# KL Divergence가 왜 '거리(distance)'가 아닌지 설명해보세요.

P_ex = [0.8, 0.15, 0.05]
Q_ex = [0.1, 0.2,  0.7 ]

# TODO:
# kl_PQ = kl_divergence(P_ex, Q_ex)
# kl_QP = kl_divergence(Q_ex, P_ex)
# print(f'D_KL(P||Q) = {kl_PQ:.4f}')
# print(f'D_KL(Q||P) = {kl_QP:.4f}')
# print(f'차이: {abs(kl_PQ - kl_QP):.4f}')


---
## 9. 주요 내용 정리

### 분류 알고리즘

| 항목 | 로지스틱 회귀 | 결정 트리 |
|------|--------------|----------|
| 결정 경계 | 선형 | 비선형 (계단형) |
| 스케일 필요 | 필요 (StandardScaler) | 불필요 |
| 해석 가능성 | coefficient 분석 | 트리 시각화 |
| 과적합 제어 | 정규화(C 파라미터) | max_depth 등 |
| 손실 함수 | Binary Cross Entropy | 지니/엔트로피 |

### 정보이론 핵심 정리

| 지표 | 수식 | 의미 | ML 활용 |
|------|------|------|--------|
| H(X) 엔트로피 | -sum p log p | 분포의 불확실성 | 결정 트리 분기 기준 |
| H(P,Q) 크로스 엔트로피 | -sum P log Q | P를 Q로 표현할 때 코드 길이 | 분류 손실 함수 |
| D_KL(P||Q) KL Divergence | sum P log(P/Q) | P, Q 간 정보 손실 | D_KL = H(P,Q) - H(P) |

### 핵심 코드 패턴

```python
# 엔트로피
H = -sum(p * np.log2(p) for p in probs if p > 0)

# 크로스 엔트로피 (손실 함수)
BCE = -np.mean(y * np.log(p) + (1-y) * np.log(1-p))  # 이진
CE  = -sum(P_i * np.log(Q_i) for P_i, Q_i in zip(P, Q))  # 다중 클래스

# KL Divergence
KL = sum(P_i * np.log(P_i / Q_i) for P_i, Q_i in zip(P, Q) if P_i > 0)
# 또는: KL = H(P,Q) - H(P)

# sklearn 활용
from sklearn.metrics import log_loss
bce = log_loss(y_test, y_prob)  # Binary/Multiclass CE Loss
```

### 다음 주: 8주차 중간고사
1~7주차 전 범위 (Python 기초, NumPy/Pandas, 시각화, 전처리, 지도학습, 정보이론) 복습!
